# Generate counterbalanced stimulus sets

Reads `latin_square_counterbalanced_sets_wide.csv` and generates one self-contained,
ready-to-run experiment folder per `(group, version)` pair (e.g. `BASE_Direct/`,
`GROUP_1_Reverse/`). Each folder gets its own 8 JS timeline files (one per rotation)
and 8 matching HTML files, built from the existing `type_update/timeline_Dis_2m_8t.js`
and `html_update/fluency.html` templates.

Only the 8 main-experiment cue words (`pos_1`...`pos_8` in the CSV) are substituted.
The practice section (instructions + the blank-word trial + the COUNTRIES trial),
`spellcheck_review`, survey/demographics, and save logic are copied verbatim from the
templates and are never touched.

Each generated page also tags its exported data with `group`/`version`/`rotation`
columns via `jsPsych.data.addProperties(...)`, so sessions from different conditions
are distinguishable later without having to cross-reference cue words.

In [1]:
import shutil
import re
import subprocess
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
CSV_PATH = BASE / "latin_square_counterbalanced_sets_wide.csv"
JS_TEMPLATE_PATH = BASE / "type_update" / "timeline_Dis_2m_8t.js"
HTML_TEMPLATE_PATH = BASE / "html_update" / "fluency.html"

assert CSV_PATH.exists(), f"CSV not found at {CSV_PATH} -- run this notebook from exp2_pilot_v2/"
assert JS_TEMPLATE_PATH.exists()
assert HTML_TEMPLATE_PATH.exists()

# Shared assets every generated page needs at runtime (dictionaries for the
# spellcheck feature, the typing-test passages, the save-data endpoint, and the
# consent screen image). Each is copied INTO every generated folder (see the
# generation loop below) so every folder is fully self-contained -- it works no
# matter what directory a static file server treats as its root, whether that's
# exp2_pilot_v2 itself or a single generated folder opened on its own (e.g. VS
# Code Live Server pointed directly at "GROUP_8_Reverse/"). A relative URL (or
# <base href="../">) can never escape a static server's own document root, so
# depending on a shared parent directory only works if the whole exp2_pilot_v2
# tree is served together -- copying the assets in removes that fragile assumption.
DICTIONARIES_DIR = BASE / "dictionaries"
TEXTS_DIR = BASE / "texts"
SAVEPHP_PATH = BASE / "SavePHP.php"
CONSENT_IMAGE_PATH = BASE / "Consent Statement for Research-1.jpg"
for p in (DICTIONARIES_DIR, TEXTS_DIR, SAVEPHP_PATH, CONSENT_IMAGE_PATH):
    assert p.exists(), f"expected shared asset at {p}"

df = pd.read_csv(CSV_PATH)
df.head()

,group,version,rotation,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8
0,BASE,Direct,1,BONNET,LINEN,TODDLER,COLOR,FUEL,SECRETARY,JEWELRY,MUSICIAN
1,BASE,Direct,2,LINEN,TODDLER,COLOR,FUEL,SECRETARY,JEWELRY,MUSICIAN,BONNET
2,BASE,Direct,3,TODDLER,COLOR,FUEL,SECRETARY,JEWELRY,MUSICIAN,BONNET,LINEN
3,BASE,Direct,4,COLOR,FUEL,SECRETARY,JEWELRY,MUSICIAN,BONNET,LINEN,TODDLER
4,BASE,Direct,5,FUEL,SECRETARY,JEWELRY,MUSICIAN,BONNET,LINEN,TODDLER,COLOR


## Load & sanity-check the CSV

Expect 9 groups (`BASE`, `GROUP_1`...`GROUP_8`) x 2 versions (`Direct`, `Reverse`) x
8 rotations = 144 rows, one `pos_1`...`pos_8` cue-word ordering per row.

In [2]:
EXPECTED_COLUMNS = ["group", "version", "rotation",
                    "pos_1", "pos_2", "pos_3", "pos_4", "pos_5", "pos_6", "pos_7", "pos_8"]
assert list(df.columns) == EXPECTED_COLUMNS, df.columns.tolist()

groups = sorted(df["group"].unique())
versions = sorted(df["version"].unique())
rotations = sorted(df["rotation"].unique())
print(f"{len(groups)} groups: {groups}")
print(f"{len(versions)} versions: {versions}")
print(f"{len(rotations)} rotations: {rotations}")
print(f"{len(df)} total rows (expect {len(groups) * len(versions) * len(rotations)})")

assert len(df) == len(groups) * len(versions) * len(rotations)
for (g, v), sub in df.groupby(["group", "version"]):
    assert sorted(sub["rotation"].tolist()) == list(range(1, 9)), f"{g}/{v} missing a rotation"
print("CSV structure OK.")

9 groups: ['BASE', 'GROUP_1', 'GROUP_2', 'GROUP_3', 'GROUP_4', 'GROUP_5', 'GROUP_6', 'GROUP_7', 'GROUP_8']
2 versions: ['Direct', 'Reverse']
8 rotations: [1, 2, 3, 4, 5, 6, 7, 8]
144 total rows (expect 144)
CSV structure OK.


## Load templates

`type_update/timeline_Dis_2m_8t.js` is the JS template (rotation 1 = trial8..trial1 in
that physical order). `html_update/fluency.html` is the HTML template. Both are only
ever *read* here -- the notebook never writes back to them.

In [3]:
js_template = JS_TEMPLATE_PATH.read_text()
html_template = HTML_TEMPLATE_PATH.read_text()
print(f"JS template: {len(js_template)} chars")
print(f"HTML template: {len(html_template)} chars")

JS template: 56255 chars
HTML template: 1995 chars


## Generator functions

Confirmed by reading the template directly: `trial8`'s cue maps to `pos_1`, `trial7`
to `pos_2`, ..., `trial1` to `pos_8` -- this is the same rotation ordering already used
by the existing `_v2.js`...`_v8.js` files, so each CSV row (already a precomputed
rotation) maps straight onto one output file with no rotation math needed.

Each trial's block is located with a regex bounded by `const trial{N} = { ... \n};`
so the substitution can't accidentally touch an unrelated occurrence of a word
elsewhere in the file. Every match count is asserted to be exactly 1 -- any drift from
what the template is expected to contain raises loudly instead of silently producing a
wrong file.

In [4]:
# trial8 -> pos_1, trial7 -> pos_2, ..., trial1 -> pos_8
TRIAL_TO_POS = {8: 1, 7: 2, 6: 3, 5: 4, 4: 5, 3: 6, 2: 7, 1: 8}

TRIAL_BLOCK_RE = {n: re.compile(r"const trial%d = \{.*?\n\};" % n, re.DOTALL) for n in range(1, 9)}
H2_RE = re.compile(r'(<h2 style="margin-bottom:10px;">)([A-Z]+)(</h2>)')
CUE_RE = re.compile(r'(cue_word: ")([A-Z]+)(")')


def build_js(template_text, positions):
    """positions: dict pos_index (1-8) -> cue word, for one CSV row."""
    content = template_text
    for trial_n in range(1, 9):
        pos_idx = TRIAL_TO_POS[trial_n]
        new_word = positions[pos_idx]

        block_re = TRIAL_BLOCK_RE[trial_n]
        matches = block_re.findall(content)
        if len(matches) != 1:
            raise ValueError(f"trial{trial_n}: expected exactly 1 block match, got {len(matches)}")
        block = matches[0]

        h2_matches = H2_RE.findall(block)
        cue_matches = CUE_RE.findall(block)
        if len(h2_matches) != 1:
            raise ValueError(f"trial{trial_n}: expected exactly 1 <h2> cue match, got {len(h2_matches)}")
        if len(cue_matches) != 1:
            raise ValueError(f"trial{trial_n}: expected exactly 1 cue_word match, got {len(cue_matches)}")
        if h2_matches[0][1] != cue_matches[0][1]:
            raise ValueError(f"trial{trial_n}: <h2> word {h2_matches[0][1]!r} != cue_word {cue_matches[0][1]!r}")

        new_block = H2_RE.sub(lambda m: m.group(1) + new_word + m.group(3), block, count=1)
        new_block = CUE_RE.sub(lambda m: m.group(1) + new_word + m.group(3), new_block, count=1)
        content = block_re.sub(lambda m: new_block, content, count=1)
    return content


SCRIPT_SRC_OLD = '<script src="type_update/timeline_Dis_2m_8t.js"></script>'
INITJSPSYCH_OLD = """    const jsPsych = initJsPsych({
      show_progress_bar: true
    });"""


BASE_HREF_OLD = '<base href="../">\n'


def build_html(template_text, js_filename, group, version, rotation):
    # Drop <base href="../"> entirely and keep every path a plain same-folder
    # filename. Since every shared asset (dictionaries/, texts/, SavePHP.php, the
    # consent image) is copied into this same folder, nothing needs to reach
    # outside it -- so the page works regardless of what a static server's
    # document root is set to.
    content = template_text

    if content.count(BASE_HREF_OLD) != 1:
        raise ValueError(f"expected exactly 1 <base href> match, got {content.count(BASE_HREF_OLD)}")
    content = content.replace(BASE_HREF_OLD, "", 1)

    if content.count(SCRIPT_SRC_OLD) != 1:
        raise ValueError(f"expected exactly 1 script-src match, got {content.count(SCRIPT_SRC_OLD)}")
    content = content.replace(SCRIPT_SRC_OLD, f'<script src="{js_filename}"></script>', 1)

    if content.count(INITJSPSYCH_OLD) != 1:
        raise ValueError(f"expected exactly 1 initJsPsych match, got {content.count(INITJSPSYCH_OLD)}")
    add_props_line = (
        f'\n    jsPsych.data.addProperties({{ group: "{group}", version: "{version}", rotation: {rotation} }});'
    )
    content = content.replace(INITJSPSYCH_OLD, INITJSPSYCH_OLD + add_props_line, 1)

    return content


print("Generator functions defined.")

Generator functions defined.


## Generate all 18 folders

One folder per `(group, version)` pair, named `{group}_{version}`, each containing 8
JS files (`timeline_Dis_2m_8t.js` for rotation 1, `..._v{rotation}.js` for 2-8) and 8
matching HTML files (`fluency.html` / `fluency_v{rotation}.html`).

In [5]:
def copy_shared_assets(folder):
    """Copy every asset the generated pages need at runtime into `folder`, so it's
    fully self-contained (see the note in the CSV-loading cell above)."""
    dst_dict = folder / "dictionaries"
    dst_dict.mkdir(exist_ok=True)
    for f in DICTIONARIES_DIR.glob("*"):
        if f.is_file() and f.suffix in (".aff", ".dic"):
            shutil.copy2(f, dst_dict / f.name)

    dst_texts = folder / "texts"
    dst_texts.mkdir(exist_ok=True)
    for f in TEXTS_DIR.glob("*"):
        if f.is_file() and f.suffix in (".csv", ".txt"):
            shutil.copy2(f, dst_texts / f.name)

    shutil.copy2(SAVEPHP_PATH, folder / SAVEPHP_PATH.name)
    shutil.copy2(CONSENT_IMAGE_PATH, folder / CONSENT_IMAGE_PATH.name)

    # SavePHP.php writes to "data/<uniqid>.csv" relative to itself; PHP does not
    # auto-create missing directories, so this must exist for saving to work.
    (folder / "data").mkdir(exist_ok=True)


written_js = []
written_html = []
written_folders = []

for (group, version), sub in df.groupby(["group", "version"], sort=False):
    folder = BASE / f"{group}_{version}"
    folder.mkdir(exist_ok=True)
    copy_shared_assets(folder)
    written_folders.append(folder)

    for _, row in sub.sort_values("rotation").iterrows():
        rotation = int(row["rotation"])
        positions = {i: row[f"pos_{i}"] for i in range(1, 9)}

        js_name = "timeline_Dis_2m_8t.js" if rotation == 1 else f"timeline_Dis_2m_8t_v{rotation}.js"
        html_name = "fluency.html" if rotation == 1 else f"fluency_v{rotation}.html"

        js_content = build_js(js_template, positions)
        html_content = build_html(html_template, js_name, group, version, rotation)

        (folder / js_name).write_text(js_content)
        (folder / html_name).write_text(html_content)

        written_js.append(folder / js_name)
        written_html.append(folder / html_name)

print(f"Wrote {len(written_js)} JS files and {len(written_html)} HTML files "
      f"across {len(written_folders)} self-contained folders "
      f"(each with its own dictionaries/, texts/, SavePHP.php, consent image, and data/).")

Wrote 144 JS files and 144 HTML files across 18 self-contained folders (each with its own dictionaries/, texts/, SavePHP.php, consent image, and data/).


## Verify

Re-read every generated JS file and confirm each `trial{N}`'s cue word matches the
source CSV row exactly; confirm every generated HTML file's script tag and
`addProperties` line are correct. Then run `node --check` on every generated JS file
to catch syntax errors.

In [6]:
errors = []

for (group, version), sub in df.groupby(["group", "version"], sort=False):
    folder = BASE / f"{group}_{version}"
    for _, row in sub.sort_values("rotation").iterrows():
        rotation = int(row["rotation"])
        positions = {i: row[f"pos_{i}"] for i in range(1, 9)}

        js_name = "timeline_Dis_2m_8t.js" if rotation == 1 else f"timeline_Dis_2m_8t_v{rotation}.js"
        html_name = "fluency.html" if rotation == 1 else f"fluency_v{rotation}.html"

        js_text = (folder / js_name).read_text()
        for trial_n in range(1, 9):
            expected = positions[TRIAL_TO_POS[trial_n]]
            block = TRIAL_BLOCK_RE[trial_n].findall(js_text)
            if len(block) != 1:
                errors.append(f"{folder/js_name}: trial{trial_n} block not found once")
                continue
            h2 = H2_RE.findall(block[0])
            cue = CUE_RE.findall(block[0])
            if not h2 or h2[0][1] != expected:
                errors.append(f"{folder/js_name}: trial{trial_n} h2 mismatch (expected {expected})")
            if not cue or cue[0][1] != expected:
                errors.append(f"{folder/js_name}: trial{trial_n} cue_word mismatch (expected {expected})")

        html_text = (folder / html_name).read_text()
        expected_src = f'<script src="{js_name}"></script>'
        if expected_src not in html_text:
            errors.append(f"{folder/html_name}: missing/incorrect script src (expected {expected_src!r})")
        if "<base href" in html_text:
            errors.append(f"{folder/html_name}: should not contain a <base href> tag")
        # Confirm every asset this page needs is actually present in ITS OWN folder --
        # this is what makes the folder work no matter what a static server's document
        # root is set to (the earlier bug: a missing local copy plus a <base href>
        # that silently clamped at whatever root was actually being served).
        for rel in ["dictionaries/en_US.aff", "dictionaries/en_US.dic", "texts/passages.csv",
                    "SavePHP.php", "Consent Statement for Research-1.jpg", "data"]:
            if not (folder / rel).exists():
                errors.append(f"{folder}: missing local copy of {rel}")
        expected_props = f'jsPsych.data.addProperties({{ group: "{group}", version: "{version}", rotation: {rotation} }});'
        if expected_props not in html_text:
            errors.append(f"{folder/html_name}: missing/incorrect addProperties line")

print(f"{len(errors)} content-verification errors.")
for e in errors[:20]:
    print(" -", e)

0 content-verification errors.


In [7]:
node_errors = []
for js_path in written_js:
    result = subprocess.run(["node", "--check", str(js_path)], capture_output=True, text=True)
    if result.returncode != 0:
        node_errors.append((str(js_path), result.stderr.strip()))

print(f"{len(node_errors)} JS files failed node --check (of {len(written_js)} total).")
for path, err in node_errors[:10]:
    print(" -", path)
    print("  ", err)

0 JS files failed node --check (of 144 total).


In [8]:
# Confirm the templates themselves were never modified.
assert JS_TEMPLATE_PATH.read_text() == js_template, "JS template was modified!"
assert HTML_TEMPLATE_PATH.read_text() == html_template, "HTML template was modified!"
print("Templates untouched. All checks passed." if not errors and not node_errors else "See errors above.")

Templates untouched. All checks passed.
